In [41]:
!pip install -r requirements.txt

In [42]:
import pandas as pd
import numpy as np
import json
import duckdb
from datetime import datetime, timedelta

In [43]:
# Define file paths
excel_file = 'Fawlty_Luxury_Collection_Data_V2.xlsx'
db_file = 'fawlty_luxury_collection_data.db'    

In [44]:
# Load the Excel file and check available sheets
xl = pd.ExcelFile(excel_file)
print('Sheets available:', xl.sheet_names)

# Load each sheet into a DataFrame
df_guests = xl.parse('Guests')
df_hotels = xl.parse('Hotels')
df_tiers = xl.parse('Tiers')
df_benefits = xl.parse('Benefits')
df_stays = xl.parse('Stays')
df_reviews = xl.parse('Reviews updated')
df_guests_tiers = xl.parse('Guests_Tiers')
df_stays_benefits = xl.parse('Stays_Benefits')
df_tiers_benefits = xl.parse('Tiers_Benefits')


print('Guests:', df_guests.shape)
print('Hotels:', df_hotels.shape)
print('Tiers:', df_tiers.shape)
print('Benefits:', df_benefits.shape)
print('Stays:', df_stays.shape)
print('Reviews:', df_reviews.shape)
print('Guests_Tiers:', df_guests_tiers.shape)
print('Stays_Benefits:', df_stays_benefits.shape)
print('Tiers_Benefits:', df_tiers_benefits.shape)

Sheets available: ['Consolidated', 'Guests', 'Hotels', 'Stays', 'Reviews updated', 'Guests_Tiers', 'Tiers', 'Benefits', 'Stays_Benefits', 'Tiers_Benefits']
Guests: (500, 8)
Hotels: (5, 8)
Tiers: (4, 3)
Benefits: (5, 3)
Stays: (4795, 11)
Reviews: (1841, 5)
Guests_Tiers: (500, 4)
Stays_Benefits: (8728, 4)
Tiers_Benefits: (11, 2)


In [45]:
# Clean & prep data
df_guests['join_date'] = pd.to_datetime(df_guests['join_date']).dt.date
df_hotels['post_code'] = df_hotels['post_code'].astype(str)
df_stays['check_in_date'] = pd.to_datetime(df_stays['check_in_date']).dt.date
df_stays['comp_night']    = df_stays['comp_night'].astype(bool)

In [46]:
# Setting up database connection
con = duckdb.connect(db_file)

# Drop tables in reverse dependency order (safe re-run)
drop_order = [
    'Stays_Benefits', 'Tiers_Benefits', 'Reviews',
    'Guests_Tiers', 'Stays', 'Guests', 'Hotels', 'Benefits', 'Tiers'
]
for table in drop_order:
    con.execute(f'DROP TABLE IF EXISTS {table}')
print('Existing tables dropped (if any).')

Existing tables dropped (if any).


In [47]:
# Create tables
con.execute("""
CREATE TABLE IF NOT EXISTS Guests (
    Guest_ID     VARCHAR PRIMARY KEY,
    first_name   VARCHAR,
    last_name    VARCHAR,
    country_code INTEGER,
    phone_number VARCHAR,
    email        VARCHAR,
    nationality  VARCHAR,
    join_date    DATE
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Hotels (
    Hotel_ID                 VARCHAR PRIMARY KEY,
    hotel_name               VARCHAR,
    address                  VARCHAR,
    post_code                VARCHAR,
    country                  VARCHAR,
    total_rooms              INTEGER,
    country_code             INTEGER,
    reception_contact_number VARCHAR
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Tiers (
    Tier_ID         VARCHAR PRIMARY KEY,
    tier_name       VARCHAR,
    nights_required INTEGER
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Benefits (
    Benefit_ID       VARCHAR PRIMARY KEY,
    benefit_name     VARCHAR,
    latest_unit_cost REAL
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Stays (
    Stay_ID          VARCHAR PRIMARY KEY,
    Guest_ID         VARCHAR REFERENCES Guests(Guest_ID),
    Hotel_ID         VARCHAR REFERENCES Hotels(Hotel_ID),
    check_in_date    DATE,
    number_of_nights INTEGER,
    room_rate        REAL,
    f_and_b_spend    REAL,
    spa_spend        REAL,
    points_earned    INTEGER,
    points_redeemed  INTEGER,
    comp_night       BOOLEAN
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Reviews (
    Review_ID            VARCHAR PRIMARY KEY,
    Stay_ID              VARCHAR REFERENCES Stays(Stay_ID),
    csat_score           INTEGER,
    review_text          VARCHAR,
    temp_review_category VARCHAR
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Guests_Tiers (
    Guest_ID            VARCHAR REFERENCES Guests(Guest_ID),
    Tier_ID             VARCHAR REFERENCES Tiers(Tier_ID),
    total_nights_stayed INTEGER,
    points_balance      INTEGER,
    PRIMARY KEY (Guest_ID, Tier_ID)
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Tiers_Benefits (
    Tier_ID     VARCHAR REFERENCES Tiers(Tier_ID),
    Benefit_ID  VARCHAR REFERENCES Benefits(Benefit_ID),
    PRIMARY KEY (Tier_ID, Benefit_ID)
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Stays_Benefits (
    Stay_ID               VARCHAR REFERENCES Stays(Stay_ID),
    Benefit_ID            VARCHAR REFERENCES Benefits(Benefit_ID),
    quantity_used_benefit INTEGER,
    historical_unit_cost  REAL,
    PRIMARY KEY (Stay_ID, Benefit_ID)
)""")

print('All tables created successfully.')

All tables created successfully.


In [48]:
# Insert in dependency order (parents before children)
con.register('df_guests', df_guests)
con.execute('INSERT INTO Guests SELECT * FROM df_guests')
print(f'Guests inserted    : {con.execute("SELECT COUNT(*) FROM Guests").fetchone()[0]} rows')

con.register('df_hotels', df_hotels)
con.execute('INSERT INTO Hotels SELECT * FROM df_hotels')
print(f'Hotels inserted    : {con.execute("SELECT COUNT(*) FROM Hotels").fetchone()[0]} rows')

con.register('df_tiers', df_tiers)
con.execute('INSERT INTO Tiers SELECT * FROM df_tiers')
print(f'Tiers inserted     : {con.execute("SELECT COUNT(*) FROM Tiers").fetchone()[0]} rows')

con.register('df_benefits', df_benefits)
con.execute('INSERT INTO Benefits SELECT * FROM df_benefits')
print(f'Benefits inserted  : {con.execute("SELECT COUNT(*) FROM Benefits").fetchone()[0]} rows')

con.register('df_stays', df_stays)
con.execute('INSERT INTO Stays SELECT * FROM df_stays')
print(f'Stays inserted     : {con.execute("SELECT COUNT(*) FROM Stays").fetchone()[0]} rows')

con.register('df_reviews', df_reviews)
con.execute('INSERT INTO Reviews SELECT * FROM df_reviews')
print(f'Reviews inserted   : {con.execute("SELECT COUNT(*) FROM Reviews").fetchone()[0]} rows')

con.register('df_guests_tiers', df_guests_tiers)
con.execute('INSERT INTO Guests_Tiers SELECT * FROM df_guests_tiers')
print(f'Guests_Tiers       : {con.execute("SELECT COUNT(*) FROM Guests_Tiers").fetchone()[0]} rows')

con.register('df_tiers_benefits', df_tiers_benefits)
con.execute('INSERT INTO Tiers_Benefits SELECT * FROM df_tiers_benefits')
print(f'Tiers_Benefits     : {con.execute("SELECT COUNT(*) FROM Tiers_Benefits").fetchone()[0]} rows')

con.register('df_stays_benefits', df_stays_benefits)
con.execute('INSERT INTO Stays_Benefits SELECT * FROM df_stays_benefits')
print(f'Stays_Benefits     : {con.execute("SELECT COUNT(*) FROM Stays_Benefits").fetchone()[0]} rows')

print('All data inserted successfully.')

Guests inserted    : 500 rows
Hotels inserted    : 5 rows
Tiers inserted     : 4 rows
Benefits inserted  : 5 rows
Stays inserted     : 4795 rows
Reviews inserted   : 1841 rows
Guests_Tiers       : 500 rows
Tiers_Benefits     : 11 rows
Stays_Benefits     : 8728 rows
All data inserted successfully.


In [49]:
# List all tables in the DB
tables = con.execute("SHOW TABLES").df()
print('Tables in database:')
display(tables)

Tables in database:


,name
0,Benefits
1,Guests
2,Guests_Tiers
3,Hotels
4,Reviews
5,Stays
6,Stays_Benefits
7,Tiers
8,Tiers_Benefits
9,df_benefits


In [50]:
# Sample query: guests with their current tier
con.execute("""
    SELECT g.Guest_ID, g.first_name, g.last_name, t.tier_name,
           gt.total_nights_stayed, gt.points_balance
    FROM Guests g
    JOIN Guests_Tiers gt ON g.Guest_ID = gt.Guest_ID
    JOIN Tiers t ON gt.Tier_ID = t.Tier_ID
    LIMIT 10
""").df()

,Guest_ID,first_name,last_name,tier_name,total_nights_stayed,points_balance
0,G987230,Nicole,Jackson,Basic,5,1178
1,G079955,Anthony,Wilson,Basic,3,583
2,G567131,Laura,Phillips,Basic,6,1243
3,G500892,James,Bailey,Basic,5,960
4,G055400,Betty,Rodriguez,Basic,4,697
5,G135050,Danielle,Jackson,Basic,6,1357
6,G693950,Beverly,Lee,Basic,7,1592
7,G732057,Timothy,Foster,Basic,7,1406
8,G051334,John,Hall,Basic,6,925
9,G995352,Eric,Cruz,Basic,2,257


In [51]:
# Sample query: average CSAT score per hotel
con.execute("""
    SELECT h.hotel_name,
           ROUND(AVG(r.csat_score), 2) AS avg_csat,
           COUNT(r.Review_ID) AS total_reviews
    FROM Reviews r
    JOIN Stays s ON r.Stay_ID = s.Stay_ID
    JOIN Hotels h ON s.Hotel_ID = h.Hotel_ID
    GROUP BY h.hotel_name
    ORDER BY avg_csat DESC
    LIMIT 10
""").df()

,hotel_name,avg_csat,total_reviews
0,Lone Star Plaza Dallas,7.06,341
1,Hudson Grand New York,6.86,443
2,Pacific Horizon Los Angeles,6.81,328
3,South Beach Vista Miami,6.81,380
4,Lakeshore Suites Chicago,6.74,349


In [52]:
# Sample query: average CSAT score per hotel
con.execute("""
    SELECT h.hotel_name,
           ROUND(AVG(r.csat_score), 2) AS avg_csat,
           COUNT(r.Review_ID)          AS total_reviews
    FROM Reviews r
    JOIN Stays s ON r.Stay_ID = s.Stay_ID
    JOIN Hotels h ON s.Hotel_ID = h.Hotel_ID
    GROUP BY h.hotel_name
    ORDER BY avg_csat DESC
    LIMIT 10
""").df()

,hotel_name,avg_csat,total_reviews
0,Lone Star Plaza Dallas,7.06,341
1,Hudson Grand New York,6.86,443
2,South Beach Vista Miami,6.81,380
3,Pacific Horizon Los Angeles,6.81,328
4,Lakeshore Suites Chicago,6.74,349


In [53]:
# Close the connection
con.close()
print(f'Database saved and connection closed: {db_file}')

Database saved and connection closed: fawlty_luxury_collection_data.db
